<a href="https://colab.research.google.com/github/HopeSilkina/deposits_forecast_project/blob/main/notebooks/03_Model_Interpretation_Deposits_Forecast_Ru.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# БЛОК 3: ИНТЕРПРЕТАЦИЯ МОДЕЛЕЙ
# Проект: Прогнозирование объема вкладов населения РФ
# Автор: Надежда Силкина
# Дата: 2026
# ============================================================

# ============================================================
# 1. ПОДКЛЮЧЕНИЕ БИБЛИОТЕК
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.model_selection import cross_val_score
import shap
import warnings
warnings.filterwarnings('ignore')

# Настройка графиков
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✅ Библиотеки загружены")

# ============================================================
# 2. ЗАГРУЗКА ДАННЫХ И ПОДГОТОВКА
# ============================================================

url = 'https://raw.githubusercontent.com/HopeSilkina/deposits_forecast_project/main/data/processed_deposits_data.xlsx'
df = pd.read_excel(url, sheet_name='data')
df['Date'] = pd.to_datetime(df['Date'])
df.set_index('Date', inplace=True)
df.sort_index(inplace=True)

# Создание признаков (лаги)
df['DEPOS_log'] = np.log(df['DEPOS'])
for lag in [1, 3, 6, 12]:
    df[f'DEPOS_lag_{lag}'] = df['DEPOS'].shift(lag)

# Подготовка X и y
X = df.drop(['DEPOS', 'DEPOS_log'], axis=1).dropna()
y = df.loc[X.index, 'DEPOS']

# Разделение на train/test
train_size = len(X) - 12
X_train, X_test = X.iloc[:train_size], X.iloc[train_size:]
y_train, y_test = y.iloc[:train_size], y.iloc[train_size:]

# Масштабирование (для Ridge и Lasso)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"✅ Данные загружены. Записей: {len(df)}")
print(f"📊 Размерность X_train: {X_train.shape}")
print(f"📊 Размерность X_test: {X_test.shape}")

# ============================================================
# 3. SHAP-АНАЛИЗ (ДЛЯ ЛУЧШЕЙ МОДЕЛИ — RIDGE)
# ============================================================

print("\n" + "="*60)
print("3. SHAP-АНАЛИЗ (RIDGE-РЕГРЕССИЯ)")
print("="*60)

ridge_model = Ridge(alpha=1.0)
ridge_model.fit(X_train_scaled, y_train)

print("\n✅ Модель Ridge обучена")

# SHAP-анализ
print("\n🔍 Расчет SHAP-значений...")
explainer = shap.LinearExplainer(ridge_model, X_train_scaled, feature_names=X.columns)
shap_values = explainer.shap_values(X_test_scaled)

# Топ-5 признаков по SHAP
shap_importance = pd.DataFrame({
    'feature': X.columns,
    'shap_importance': np.abs(shap_values).mean(axis=0)
}).sort_values('shap_importance', ascending=False)

print("\n📊 ТОП-5 ПРИЗНАКОВ ПО SHAP (RIDGE):")
print(shap_importance.head(5).to_string(index=False))

# 3.3. Визуализация SHAP
print("\n📊 Визуализация SHAP...")

# График 1: Глобальная важность признаков
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test_scaled, feature_names=X.columns, plot_type="bar", show=False)
plt.title('Глобальная важность признаков (SHAP)', fontsize=14)
plt.tight_layout()
plt.savefig('03_shap_importance.png', dpi=300, bbox_inches='tight')
plt.show()

# График 2: Направление влияния
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test_scaled, feature_names=X.columns, show=False)
plt.title('Направление влияния признаков (SHAP)', fontsize=14)
plt.tight_layout()
plt.savefig('03_shap_summary.png', dpi=300, bbox_inches='tight')
plt.show()

# График 3: Зависимость SHAP от значения признака (для топ-1)
top_feature = shap_importance.iloc[0]['feature']
plt.figure(figsize=(8, 6))
shap.dependence_plot(top_feature, shap_values, X_test_scaled, feature_names=X.columns, show=False)
plt.title(f'Зависимость SHAP от {top_feature}', fontsize=14)
plt.tight_layout()
plt.savefig('03_shap_dependence.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✅ SHAP-анализ завершен")

# ============================================================
# 4. АНАЛИЗ НЕЛИНЕЙНЫХ СВЯЗЕЙ
# ============================================================

print("\n" + "="*60)
print("4. АНАЛИЗ НЕЛИНЕЙНЫХ СВЯЗЕЙ")
print("="*60)

def calculate_full_metrics(y_train, y_test, y_train_pred, y_test_pred, n_params):
    """
    Calculate comprehensive metrics on both training and test sets.

    Parameters:
    -----------
    y_train, y_test : array-like
        Actual values
    y_train_pred, y_test_pred : array-like
        Predicted values
    n_params : int
        Number of model parameters (including intercept)

    Returns:
    --------
    dict with metrics including overfitting gap
    """
    n_train = len(y_train)

    # Training set metrics
    r2_train = r2_score(y_train, y_train_pred)
    rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))

    # Adjusted R² on training set
    if n_train - n_params - 1 > 0:
        r2_adj_train = 1 - (1 - r2_train) * (n_train - 1) / (n_train - n_params - 1)
    else:
        r2_adj_train = np.nan

    # AIC and BIC on training set (unbiased variance estimator)
    residuals_train = y_train - y_train_pred
    rss_train = np.sum(residuals_train**2)
    sigma2 = rss_train / (n_train - n_params)
    log_likelihood = -0.5 * n_train * (np.log(2 * np.pi * sigma2) + 1)

    aic = -2 * log_likelihood + 2 * n_params
    bic = -2 * log_likelihood + n_params * np.log(n_train)

    # Test set metrics
    r2_test = r2_score(y_test, y_test_pred)
    rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred))

    # Overfitting gap: R²_train - R²_test
    r2_gap = r2_train - r2_test

    return {
        'n_params': n_params,
        'r2_train': r2_train,
        'r2_adj_train': r2_adj_train,
        'aic': aic,
        'bic': bic,
        'r2_test': r2_test,
        'rmse_test': rmse_test,
        'r2_gap': r2_gap  # Разница R²_train - R²_test (чем больше, тем сильнее переобучение)
    }

# ============================================================
# 4.1. Базовая модель (Linear Regression)
# ============================================================

print("\n🔍 Базовая линейная модель (для сравнения):")
lr_base = LinearRegression()
lr_base.fit(X_train, y_train)

y_train_pred_base = lr_base.predict(X_train)
y_test_pred_base = lr_base.predict(X_test)

n_params_base = X_train.shape[1] + 1
metrics_base = calculate_full_metrics(
    y_train, y_test,
    y_train_pred_base, y_test_pred_base,
    n_params_base
)

print(f"   R²_train = {metrics_base['r2_train']:.4f}, R²_test = {metrics_base['r2_test']:.4f}")
print(f"   (полные метрики — в сравнительной таблице ниже)")

# ============================================================
# 4.2. Тест Рамсея (RESET test)
# ============================================================

print("\n🔍 Тест Рамсея (проверка на пропущенные нелинейности)...")
print("   Метод: добавляем y_pred² и y_pred³ в модель и проверяем их значимость")

# Добавляем степени предсказанных значений
X_train_reset = X_train.copy()
X_train_reset['y_pred2'] = y_train_pred_base ** 2
X_train_reset['y_pred3'] = y_train_pred_base ** 3

# Обучаем модель с добавленными степенями
lr_reset = LinearRegression()
lr_reset.fit(X_train_reset, y_train)

# Проверяем значимость добавленных признаков
from sklearn.feature_selection import f_regression
f_values, p_values = f_regression(X_train_reset[['y_pred2', 'y_pred3']], y_train)

print(f"\n📊 Результаты теста Рамсея:")
print(f"   y_pred²: F = {f_values[0]:.4f}, p = {p_values[0]:.4f}")
print(f"   y_pred³: F = {f_values[1]:.4f}, p = {p_values[1]:.4f}")

if p_values[0] < 0.05 or p_values[1] < 0.05:
    print("   📌 ВЫВОД: Пропущенные нелинейности обнаружены (p < 0.05)")
    print("   → Проверяем, улучшают ли полиномиальные признаки качество модели")
else:
    print("   📌 ВЫВОД: Пропущенные нелинейности не обнаружены (p >= 0.05)")

# ============================================================
# 4.3. Обучение альтернативных моделей
# ============================================================

print("\n" + "-"*60)
print("ОБУЧЕНИЕ АЛЬТЕРНАТИВНЫХ МОДЕЛЕЙ")
print("-"*60)

# --- Модель 1: Все полиномы (degree=2) ---
print("\n🔧 Модель 1: Все полиномы (degree=2)")
print("   Описание: Все квадратичные признаки и взаимодействия (104 признака)")

poly_all = PolynomialFeatures(degree=2, include_bias=False, interaction_only=False)
X_train_poly_all = poly_all.fit_transform(X_train)
X_test_poly_all = poly_all.transform(X_test)

lr_poly_all = LinearRegression()
lr_poly_all.fit(X_train_poly_all, y_train)

y_train_pred_poly = lr_poly_all.predict(X_train_poly_all)
y_test_pred_poly = lr_poly_all.predict(X_test_poly_all)

metrics_poly = calculate_full_metrics(
    y_train, y_test,
    y_train_pred_poly, y_test_pred_poly,
    X_train_poly_all.shape[1] + 1
)

# --- Модель 2: Lasso-регуляризация ---
print("\n🔧 Модель 2: Lasso-регуляризация")
print("   Описание: L1-регуляризация для автоматического отбора признаков из полиномов")

lasso = Lasso(alpha=0.01, max_iter=10000)
lasso.fit(X_train_poly_all, y_train)

n_nonzero = np.sum(np.abs(lasso.coef_) > 1e-6)
print(f"   Отобрано признаков: {n_nonzero} из {X_train_poly_all.shape[1]}")

y_train_pred_lasso = lasso.predict(X_train_poly_all)
y_test_pred_lasso = lasso.predict(X_test_poly_all)

metrics_lasso = calculate_full_metrics(
    y_train, y_test,
    y_train_pred_lasso, y_test_pred_lasso,
    n_nonzero + 1  # только ненулевые коэффициенты + intercept
)

# --- Модель 3: Гипотезы из EDA ---
print("\n🔧 Модель 3: Признаки из EDA-гипотез")
print("   Описание: DEP1², UNEM², UNEM×DEP1, SERV² (17 признаков)")

X_train_hyp = X_train.copy()
X_test_hyp = X_test.copy()

X_train_hyp['DEP1_sq'] = X_train_hyp['DEP1'] ** 2
X_test_hyp['DEP1_sq'] = X_test_hyp['DEP1'] ** 2

X_train_hyp['UNEM_sq'] = X_train_hyp['UNEM'] ** 2
X_test_hyp['UNEM_sq'] = X_test_hyp['UNEM'] ** 2

X_train_hyp['UNEM_DEP1'] = X_train_hyp['UNEM'] * X_train_hyp['DEP1']
X_test_hyp['UNEM_DEP1'] = X_test_hyp['UNEM'] * X_test_hyp['DEP1']

X_train_hyp['SERV_sq'] = X_train_hyp['SERV'] ** 2
X_test_hyp['SERV_sq'] = X_test_hyp['SERV'] ** 2

lr_hyp = LinearRegression()
lr_hyp.fit(X_train_hyp, y_train)

y_train_pred_hyp = lr_hyp.predict(X_train_hyp)
y_test_pred_hyp = lr_hyp.predict(X_test_hyp)

metrics_hyp = calculate_full_metrics(
    y_train, y_test,
    y_train_pred_hyp, y_test_pred_hyp,
    X_train_hyp.shape[1] + 1
)

# --- Модель 4: Stepwise Selection ---
print("\n🔧 Модель 4: Пошаговый отбор (Stepwise Selection)")
print("   Описание: Последовательный отбор 10 лучших признаков с кросс-валидацией")

sfs = SequentialFeatureSelector(
    LinearRegression(),
    n_features_to_select=10,
    direction='forward',
    scoring='r2',
    cv=5,
    n_jobs=-1
)

sfs.fit(X_train_hyp, y_train)
selected_features = X_train_hyp.columns[sfs.get_support()].tolist()

print(f"   Отобрано признаков: {len(selected_features)}")
print(f"   Признаки: {selected_features}")

X_train_sfs = X_train_hyp[selected_features]
X_test_sfs = X_test_hyp[selected_features]

lr_sfs = LinearRegression()
lr_sfs.fit(X_train_sfs, y_train)

y_train_pred_sfs = lr_sfs.predict(X_train_sfs)
y_test_pred_sfs = lr_sfs.predict(X_test_sfs)

metrics_sfs = calculate_full_metrics(
    y_train, y_test,
    y_train_pred_sfs, y_test_pred_sfs,
    X_train_sfs.shape[1] + 1
)

# ============================================================
# 4.4. СРАВНИТЕЛЬНАЯ ТАБЛИЦА ВСЕХ МОДЕЛЕЙ
# ============================================================

print("\n" + "="*60)
print("📊 СРАВНИТЕЛЬНАЯ ТАБЛИЦА ВСЕХ МОДЕЛЕЙ")
print("="*60)

models_comparison = pd.DataFrame({
    'Модель': [
        'Базовая (Linear)',
        'Все полиномы',
        'Lasso (отбор)',
        'Гипотезы EDA',
        'Stepwise (отбор)'
    ],
    'Признаков': [
        metrics_base['n_params'] - 1,
        metrics_poly['n_params'] - 1,
        metrics_lasso['n_params'] - 1,
        metrics_hyp['n_params'] - 1,
        metrics_sfs['n_params'] - 1
    ],
    'R²_train': [
        metrics_base['r2_train'],
        metrics_poly['r2_train'],
        metrics_lasso['r2_train'],
        metrics_hyp['r2_train'],
        metrics_sfs['r2_train']
    ],
    'R²_adj_train': [
        metrics_base['r2_adj_train'],
        metrics_poly['r2_adj_train'],
        metrics_lasso['r2_adj_train'],
        metrics_hyp['r2_adj_train'],
        metrics_sfs['r2_adj_train']
    ],
    'AIC': [
        metrics_base['aic'],
        metrics_poly['aic'],
        metrics_lasso['aic'],
        metrics_hyp['aic'],
        metrics_sfs['aic']
    ],
    'BIC': [
        metrics_base['bic'],
        metrics_poly['bic'],
        metrics_lasso['bic'],
        metrics_hyp['bic'],
        metrics_sfs['bic']
    ],
    'R²_test': [
        metrics_base['r2_test'],
        metrics_poly['r2_test'],
        metrics_lasso['r2_test'],
        metrics_hyp['r2_test'],
        metrics_sfs['r2_test']
    ],
    'RMSE_test': [
        metrics_base['rmse_test'],
        metrics_poly['rmse_test'],
        metrics_lasso['rmse_test'],
        metrics_hyp['rmse_test'],
        metrics_sfs['rmse_test']
    ],
    'R²_gap': [
        metrics_base['r2_gap'],
        metrics_poly['r2_gap'],
        metrics_lasso['r2_gap'],
        metrics_hyp['r2_gap'],
        metrics_sfs['r2_gap']
    ]
})

print("\n📊 Метрики на ОБУЧАЮЩЕЙ выборке (n=122):")
print(models_comparison[['Модель', 'Признаков', 'R²_train', 'R²_adj_train', 'AIC', 'BIC']].round(4).to_string(index=False))

print("\n📊 Метрики на ТЕСТОВОЙ выборке (n=12):")
print(models_comparison[['Модель', 'R²_test', 'RMSE_test', 'R²_gap']].round(4).to_string(index=False))

print("\n📌 Примечание: R²_gap = R²_train - R²_test — разница между качеством")
print("   на обучающей и тестовой выборках. Большой R²_gap указывает")
print("   на переобучение модели.")

# ============================================================
# 4.5. ПРОВЕРКА КАЖДОЙ ГИПОТЕЗЫ ПО ОТДЕЛЬНОСТИ
# ============================================================

print("\n" + "-"*60)
print("ДЕТАЛЬНАЯ ПРОВЕРКА ГИПОТЕЗ ИЗ EDA")
print("-"*60)
print("Каждая гипотеза проверяется отдельно: добавляем один признак")
print("к базовой модели и смотрим изменение метрик.")

def test_hypothesis_full(feature_name, y_train, y_test, metrics_base):
    """
    Test a single hypothesis by adding one feature to base model.
    """
    X_train_test = X_train.copy()
    X_test_test = X_test.copy()
    X_train_test[feature_name] = X_train_hyp[feature_name]
    X_test_test[feature_name] = X_test_hyp[feature_name]

    lr_test = LinearRegression()
    lr_test.fit(X_train_test, y_train)

    y_train_pred_test = lr_test.predict(X_train_test)
    y_test_pred_test = lr_test.predict(X_test_test)

    n_params_test = X_train_test.shape[1] + 1
    metrics_test = calculate_full_metrics(
        y_train, y_test,
        y_train_pred_test, y_test_pred_test,
        n_params_test
    )

    return {
        'feature': feature_name,
        'r2_train_improvement': metrics_test['r2_train'] - metrics_base['r2_train'],
        'r2_test_improvement': metrics_test['r2_test'] - metrics_base['r2_test'],
        'aic_change': metrics_test['aic'] - metrics_base['aic'],
        'r2_gap_change': metrics_test['r2_gap'] - metrics_base['r2_gap']
    }

hypotheses = {
    'DEP1²': 'DEP1_sq',
    'UNEM²': 'UNEM_sq',
    'UNEM × DEP1': 'UNEM_DEP1',
    'SERV²': 'SERV_sq'
}

hypothesis_results = []
for name, feature in hypotheses.items():
    result = test_hypothesis_full(feature, y_train, y_test, metrics_base)
    result['hypothesis_name'] = name
    hypothesis_results.append(result)

# Создаем DataFrame для наглядности
hyp_df = pd.DataFrame(hypothesis_results)
hyp_df = hyp_df[['hypothesis_name', 'feature', 'r2_train_improvement', 'r2_test_improvement', 'aic_change', 'r2_gap_change']]
hyp_df.columns = ['Гипотеза', 'Признак', 'ΔR²_train', 'ΔR²_test', 'ΔAIC', 'ΔR²_gap']

print("\n📊 Результаты проверки гипотез:")
print(hyp_df.round(4).to_string(index=False))

print("\n📌 Интерпретация:")
print("   ΔR²_train > 0 — модель лучше объясняет обучающие данные")
print("   ΔR²_test > 0 — модель лучше предсказывает новые данные")
print("   ΔAIC < 0 — модель лучше с учетом сложности (штраф за параметры)")
print("   ΔR²_gap > 0 — увеличилось переобучение")

# Текстовые пояснения для каждой гипотезы
print("\n📝 Выводы по каждой гипотезе:")
for _, row in hyp_df.iterrows():
    name = row['Гипотеза']
    dr2_train = row['ΔR²_train']
    dr2_test = row['ΔR²_test']
    daic = row['ΔAIC']
    dgap = row['ΔR²_gap']

    print(f"\n   {name}:")
    if dr2_test > 0.001 and daic < 0:
        print(f"      ✅ ПОДТВЕРЖДЕНА: улучшает и прогноз (ΔR²_test = {dr2_test:+.4f}),")
        print(f"         и информационный критерий (ΔAIC = {daic:+.2f})")
    elif dr2_test > 0.001:
        print(f"      ⚠️ ЧАСТИЧНО: улучшает прогноз (ΔR²_test = {dr2_test:+.4f}),")
        print(f"         но AIC ухудшился (ΔAIC = {daic:+.2f}) — рост качества")
        print(f"         не компенсирует усложнение модели")
    elif dr2_train > 0.001 and dr2_test <= 0.001:
        print(f"      ❌ НЕ ПОДТВЕРЖДЕНА: улучшает только обучающую выборку")
        print(f"         (ΔR²_train = {dr2_train:+.4f}), но не тестовую")
        print(f"         (ΔR²_test = {dr2_test:+.4f}) — признак не обобщается")
    else:
        print(f"      ❌ НЕ ПОДТВЕРЖДЕНА: не дает улучшения ни на одной выборке")
        print(f"         (ΔR²_train = {dr2_train:+.4f}, ΔR²_test = {dr2_test:+.4f})")

# ============================================================
# 4.6. ИТОГОВЫЙ ВЫВОД
# ============================================================

print("\n" + "="*60)
print("📌 ИТОГОВЫЙ ВЫВОД ПО АНАЛИЗУ НЕЛИНЕЙНОСТЕЙ")
print("="*60)

# Выбор лучшей модели по разным критериям
best_by_r2_train = models_comparison.loc[models_comparison['R²_train'].idxmax()]
best_by_r2_test = models_comparison.loc[models_comparison['R²_test'].idxmax()]
best_by_aic = models_comparison.loc[models_comparison['AIC'].idxmin()]
best_by_bic = models_comparison.loc[models_comparison['BIC'].idxmin()]

print(f"\n🏆 Лучшие модели по разным критериям:")
print(f"   По R²_train (качество подгонки): {best_by_r2_train['Модель']} ({best_by_r2_train['R²_train']:.4f})")
print(f"   По R²_test (прогнозная способность): {best_by_r2_test['Модель']} ({best_by_r2_test['R²_test']:.4f})")
print(f"   По AIC (баланс качество/сложность): {best_by_aic['Модель']} ({best_by_aic['AIC']:.2f})")
print(f"   По BIC (с усиленным штрафом): {best_by_bic['Модель']} ({best_by_bic['BIC']:.2f})")

print(f"\n📊 Анализ переобучения (R²_gap = R²_train - R²_test):")
for _, row in models_comparison.iterrows():
    gap = row['R²_gap']
    if gap > 5.0:
        status = "❌ КРИТИЧЕСКОЕ"
    elif gap > 1.0:
        status = "❌ СИЛЬНОЕ"
    elif gap > 0.2:
        status = "⚠️ Умеренное"
    elif gap > 0.1:
        status = "ℹ️ Небольшое"
    else:
        status = "✅ Хорошее обобщение"
    print(f"   {row['Модель']}: gap = {gap:.4f} → {status}")

print(f"\n📌 КЛЮЧЕВЫЕ ВЫВОДЫ:")
print(f"   1. Тест Рамсея обнаружил нелинейности (p < 0.05)")
print(f"   2. Но добавление полиномов приводит к КРИТИЧЕСКОМУ переобучению:")
print(f"      - Все полиномы: R²_train = {metrics_poly['r2_train']:.4f}, R²_test = {metrics_poly['r2_test']:.4f}")
print(f"      - Lasso: R²_train = {metrics_lasso['r2_train']:.4f}, R²_test = {metrics_lasso['r2_test']:.4f}")
print(f"   3. Гипотезы из EDA не улучшают прогноз:")
print(f"      - UNEM² и UNEM×DEP1 дают слабое улучшение R²_test, но ухудшают AIC")
print(f"      - DEP1² и SERV² не дают улучшения")
print(f"   4. Базовая линейная модель остается оптимальной по R²_test = {metrics_base['r2_test']:.4f}")
print(f"   5. Нелинейности существуют, но их прямое добавление в модель")
print(f"      приводит к переобучению из-за малого размера выборки (n=122)")

print("\n✅ Блок 3 завершен")